# F0_plan (local CPU): pool-half evaluation set, training rows for E2/E3, hashes
Writes `FU/plan.json`, `FU/inputs/eval_half.jsonl`, then `FU/plan.DONE`.

In [ ]:
FORCE = False


In [ ]:
import glob, hashlib, json, os, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd

MYDRIVE = Path(glob.glob(os.path.expanduser("~/Library/CloudStorage/GoogleDrive-*"))[0]) / "My Drive"
PT3 = MYDRIVE / "ez-penta" / "post_train_3"
FU = PT3 / "followups"
FU.mkdir(parents=True, exist_ok=True)


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def load_jsonl(p):
    p = Path(p)
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()] if p.exists() else []


def write_jsonl(path, rs):
    with open(path, "w", encoding="utf-8") as f:
        for r in rs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def folder_ok(d):
    d = Path(d)
    if not (d / "DONE").exists() or not (d / "manifest.json").exists():
        return False
    if (d / "DONE").read_text().strip() != sha256_file(d / "manifest.json"):
        return False
    man = json.loads((d / "manifest.json").read_text())
    return all((d / f).exists() and sha256_file(d / f) == m["sha256"] for f, m in man["files"].items())


In [ ]:
# 1. Inputs and their hashes (training items, frozen pool, batches, Japan chunks, bilingual plan and eval sets).
ARM = PT3 / "arm_en"
JC = PT3 / "japan_culture"
BI = PT3 / "bilingual"
assert not ((FU / "plan.DONE").exists() and not FORCE), "plan.DONE exists"
bplan = json.loads((BI / "plan.json").read_text())
assert (BI / "plan.DONE").read_text().strip() == sha256_file(BI / "plan.json")
train_ids = json.loads((ARM / "train" / "train_item_ids.json").read_text())
assert train_ids == bplan["train_item_ids"] and len(train_ids) == 568
assert sha256_file(ARM / "train" / "items.jsonl") == bplan["hashes"]["items.jsonl"]
items = {r["id"]: r for r in load_jsonl(ARM / "train" / "items.jsonl")}
evp_all = load_jsonl(BI / "inputs" / "eval_prompts.jsonl")
assert sha256_file(BI / "inputs" / "eval_prompts.jsonl") == bplan["inputs"]["eval_prompts.jsonl"]["sha256"] and len(evp_all) == 12426
fb = bplan["eval"]["first_batch_of_seed"]
HALF_BATCHES = [1, 3, 5]
half_seeds = sorted(s for s, b in fb.items() if b in HALF_BATCHES)
print("pool-half seeds:", len(half_seeds))
assert len(half_seeds) == 97 + 89 + 90
for c in HALF_BATCHES:
    for subd in ("prompts", "base", "en_s0"):
        assert folder_ok(JC / "chunks" / f"c{c:04d}" / subd), (c, subd)
checks = []
for c in HALF_BATCHES:
    checks += load_jsonl(JC / "chunks" / f"c{c:04d}" / "prompts" / "seed_checks.jsonl")
paired_ok_half = sorted(c["seed_id"] for c in checks if c["paired_ok"])
print("paired_ok seeds in the half:", len(paired_ok_half))


In [ ]:
# 2. Pool-half evaluation prompts (US + Japan), with batch numbers for checkpointing.
hs = set(half_seeds)
evh = [dict(p, batch=fb[p["seed_id"]]) for p in evp_all if p["seed_id"] in hs]
from collections import Counter
print(Counter(p["culture"] for p in evh))
assert sum(p["culture"] == "us" for p in evh) == 12 * len(half_seeds)


In [ ]:
# 3. Training rows (item id, language) for the new conditions, in train_item_ids order.
kind = {i: items[i]["item_kind"] for i in train_ids}
assert set(kind.values()) == {"refusal", "helpful"}
half_of = {i: ("A" if int(hashlib.sha256(i.split("|")[0].encode()).hexdigest(), 16) % 2 == 0 else "B") for i in train_ids}
rows = {
    "bi_h2": [[i, l] for i in train_ids for l in (("en", "ja") if kind[i] == "refusal" else ("en", "ja", "en", "ja"))],
    "ja2": [[i, l] for i in train_ids for l in ("ja", "ja")],
    "en_A": [[i, "en"] for i in train_ids if half_of[i] == "A"],
    "ja_A": [[i, "ja"] for i in train_ids if half_of[i] == "A"],
    "en_B": [[i, "en"] for i in train_ids if half_of[i] == "B"],
    "ja_B": [[i, "ja"] for i in train_ids if half_of[i] == "B"],
}
order_text = {"bi_h2": "per item: refusal -> English row, Japanese row; helpful -> English, Japanese, English, Japanese",
              "ja2": "per item: Japanese row twice", "en_A": "English rows of item half A", "ja_A": "Japanese rows of item half A",
              "en_B": "English rows of item half B", "ja_B": "Japanese rows of item half B"}
seedsA = {i.split("|")[0] for i in train_ids if half_of[i] == "A"}
seedsB = {i.split("|")[0] for i in train_ids if half_of[i] == "B"}
assert not (seedsA & seedsB)
for k, v in rows.items():
    print(k, len(v), Counter(kind[i] for i, _ in v), Counter(l for _, l in v))
CONDS = {"bi_h2": [0, 1, 2], "ja2": [0, 1, 2], "en_A": [0], "ja_A": [0], "en_B": [0], "ja_B": [0]}


In [ ]:
# 4. Write inputs and plan.json, then plan.DONE.
(FU / "inputs").mkdir(parents=True, exist_ok=True)
write_jsonl(FU / "inputs" / "eval_half.jsonl", evh)
plan = {"created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "half_batches": HALF_BATCHES, "half_seeds": half_seeds,
        "paired_ok_half": paired_ok_half, "n_eval_half": len(evh), "conds": CONDS, "rows": rows, "row_order": order_text,
        "row_counts": {k: len(v) for k, v in rows.items()}, "item_half": half_of,
        "hashes": {"items.jsonl": sha256_file(ARM / "train" / "items.jsonl"), "train_items_sha256": bplan["hashes"]["train_items_sha256"],
                   "bilingual_plan": sha256_file(BI / "plan.json"), "eval_half.jsonl": sha256_file(FU / "inputs" / "eval_half.jsonl")}}
(FU / "plan.json").write_text(json.dumps(plan, indent=1, ensure_ascii=False))
(FU / "plan.DONE").write_text(sha256_file(FU / "plan.json") + "\n")
print("plan written:", plan["row_counts"], plan["n_eval_half"])
